# Agent State

**State** is the mutable data an agent carries through a conversation. Unlike runtime context (read-only, passed in per call), state can be **written to** by tools and **read back** later, and it persists across turns when a checkpointer is used.

In this notebook you'll:
1. Extend the built-in `AgentState` with a custom field.
2. Write to state from a tool using `Command`.
3. Read state back from another tool via `ToolRuntime`.

**Key learnings**
- State is mutable and evolves across the conversation
- Custom state schemas extend AgentState
- Tools can update state with Command


### Load environment variables

`load_dotenv()` initializes the environment by loading the `.env` file so the model provider API key is available.

In [ ]:
from dotenv import load_dotenv

load_dotenv()

### Define a custom state schema

We subclass `AgentState` (which already tracks `messages`) and add a `favourite_colour` field. This `CustomState` becomes the schema the agent uses, so tools can store and retrieve the user's favourite colour alongside the conversation.

In [ ]:
from langchain.agents import AgentState

class CustomState(AgentState):
    favourite_colour: str

## Write to state

Tools can update state by returning a `Command` with an `update` payload. This section builds a tool that records the user's favourite colour into state.

### Define a tool that writes to state

`update_favourite_colour` returns a `Command` whose `update` sets the `favourite_colour` field and appends a `ToolMessage` confirming the update. The `tool_call_id` (from `runtime`) links the message back to the specific tool call so the agent can pair the result with its request.

In [ ]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def update_favourite_colour(favourite_colour: str, runtime: ToolRuntime) -> Command:
    """Update the favourite colour of the user in the state once they've revealed it."""
    return Command(update={
        "favourite_colour": favourite_colour, 
        "messages": [ToolMessage("Successfully updated favourite colour", tool_call_id=runtime.tool_call_id)]}
        )

### Create the agent with the custom state

We build the agent with the write tool, an `InMemorySaver` checkpointer (so state persists across turns per `thread_id`), and `state_schema=CustomState` so the agent knows about the `favourite_colour` field.

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    "gpt-5-nano",
    tools=[update_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

### Reveal a favourite colour

The user says "My favourite colour is green." The agent should call `update_favourite_colour`, which writes `green` into state for thread `1`.

In [ ]:
from langchain.messages import HumanMessage

response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

### Inspect the response

`pprint` shows the message history, including the tool call that updated the state.

In [ ]:
from pprint import pprint

pprint(response)

### Seed state directly at invoke time

State can also be provided directly in the input, not just written by tools. Here we pass `favourite_colour: "green"` alongside the message on a fresh thread (`10`), pre-populating the state without any tool call.

In [ ]:
response = agent.invoke(
    { 
        "messages": [HumanMessage(content="Hello, how are you?")],
        "favourite_colour": "green"
    },
    {"configurable": {"thread_id": "10"}}
)

pprint(response)

## Read state

Writing is only half the story. Tools can also **read** from state via `runtime.state`, letting the agent recall information saved earlier in the conversation.

### Define a read tool and rebuild the agent

`read_favourite_colour` looks up `favourite_colour` from `runtime.state`, returning a fallback message if it hasn't been set yet. We then recreate the agent with **both** the write and read tools so it can store and later retrieve the colour.

In [ ]:
@tool
def read_favourite_colour(runtime: ToolRuntime) -> str:
    """Read the favourite colour of the user from the state."""
    try:
        return runtime.state["favourite_colour"]
    except KeyError:
        return "No favourite colour found in state"

agent = create_agent(
    "gpt-5-nano",
    tools=[update_favourite_colour, read_favourite_colour],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

### Write the colour on thread 1

First we tell the agent the favourite colour is green on thread `1`, which triggers the write tool and stores it in that thread's state.

In [ ]:
response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

### Read the colour back on the same thread

On the same thread `1`, we ask "What's my favourite colour?". Because the checkpointer persisted the state, the agent calls `read_favourite_colour` and recalls green from earlier in the conversation.

In [ ]:
response = agent.invoke(
    { "messages": [HumanMessage(content="What's my favourite colour?")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)